In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import pandas as pd
import glob
import numpy as np

from torch.utils.data import Dataset, DataLoader, TensorDataset, ConcatDataset

In [11]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Uncomment the following line if you're a mac (mps) user
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"Using {device} device")

Using mps device


In [12]:
PREDICTIONS_H = 5
INPUT_LENGTH = 25
LEARNING_RATE = 1e-3

In [13]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1, bidirectional=True):
        super().__init__()

        self.data_in = nn.Linear(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers,
                            bidirectional=True, batch_first=True)
        self.bidirectional = bidirectional
        self.hidden_dim = hidden_dim

    def forward(self, x):
        emb = self.data_in(x)
        outputs, (h_n, c_n) = self.lstm(emb)
        return outputs, (h_n, c_n)

class AttentionModule(nn.Module):
    def __init__(self, encoding_dim, decoding_dim, attention_dim):
        super().__init__()
        self.enc = nn.Linear(encoding_dim, attention_dim, bias=False)
        self.dec = nn.Linear(decoding_dim, attention_dim)
        self.v = nn.Linear(attention_dim, 1, bias=False)
        self._warned = False

    def forward(self, dec_hidden, encoding_outputs, mask=None):
        if encoding_outputs.dim() == 2:
            encoding_outputs = encoding_outputs.unsqueeze(0)
        if dec_hidden.dim() == 1:
            dec_hidden = dec_hidden.unsqueeze(0)

        # Project encoder and decoder to attention space
        enc_proj = self.enc(encoding_outputs)
        dec_proj = self.dec(dec_hidden).unsqueeze(1) 

        # Compute raw scores -> [B, seq_len, 1] -> squeeze -> [B, seq_len]
        scores = self.v(torch.tanh(enc_proj + dec_proj)).squeeze(-1)

        # Mask handling 
        if mask is not None:
            if mask.dim() == 1:
                mask = mask.unsqueeze(0)
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = F.softmax(scores, dim=1)

        # --- Ensure shapes for bmm ---
        if weights.dim() == 1:
            weights = weights.unsqueeze(0)
        weights_bmm = weights.unsqueeze(1) if weights.dim() == 2 else weights

        # encoding_outputs must be [B, seq_len, enc_dim]
        if encoding_outputs.dim() != 3:
            encoding_outputs = encoding_outputs.unsqueeze(0)

        # Final safety check 
        if weights_bmm.dim() != 3 or encoding_outputs.dim() != 3:
            if not self._warned:
                print("AttentionModule error: shapes before bmm:",
                      "weights_bmm.shape=", getattr(weights_bmm, "shape", None),
                      "encoding_outputs.shape=", getattr(encoding_outputs, "shape", None))
                self._warned = True
            raise RuntimeError(f"AttentionModule: expected 3D tensors for bmm but got "
                               f"{weights_bmm.dim()}D and {encoding_outputs.dim()}D tensors.")
        context = torch.bmm(weights_bmm, encoding_outputs).squeeze(1)

        return context, weights

class Decoder(nn.Module):
    def __init__(self, encoding_dim, decoding_dim, attention_dim, out_dim=1, emb_dim=32):
        super().__init__()

        self.attnMod = AttentionModule(encoding_dim, decoding_dim, attention_dim) 
        self.lstmCell = nn.LSTMCell(encoding_dim + emb_dim, decoding_dim)
        self.input_proj = nn.Linear(1, emb_dim)
        self.out = nn.Linear(decoding_dim + encoding_dim, out_dim)

    def forward(self, encoding_outputs, dec_h, dec_c, iters, first_input=None, mask=None):
        batch = encoding_outputs.size(0)
        device = encoding_outputs.device
        predictions = []
        input_t = first_input if first_input is not None else torch.zeros(batch, 1, device=device)

        for i in range(iters):
            emb_in = self.input_proj(input_t)
            context, weights = self.attnMod(dec_h, encoding_outputs, mask)
            lstm_in = torch.cat([emb_in, context], dim=1)
            dec_h, dec_c = self.lstmCell(lstm_in, (dec_h, dec_c))
            out = self.out(torch.cat([dec_h, context], dim=1))
            predictions.append(out)
            input_t = out.detach()

        return predictions

class Wrapper(nn.Module):
    def __init__(self, input_dim, enc_emb, enc_hid, dec_hid, attn_dim, bidir=True):
        super().__init__()

        self.encoder = Encoder(input_dim, enc_emb, enc_hid, bidirectional=bidir)
        self.enc_dim = enc_hid * (2 if bidir else 1)
        self.h_proj = nn.Linear(self.enc_dim * 1, dec_hid)
        self.c_proj = nn.Linear(self.enc_dim * 1, dec_hid)
        self.decoder = Decoder(self.enc_dim, dec_hid, attn_dim, out_dim=1)
        self.H = PREDICTIONS_H

    def forward(self, x, first_input=None):
        enc_out, (h_n, c_n) = self.encoder(x)
        batch = x.size(0)
        h_flat = h_n.permute(1, 0, 2).contiguous().view(batch, -1)
        c_flat = c_n.permute(1, 0, 2).contiguous().view(batch, -1)
        dec_h = torch.tanh(self.h_proj(h_flat))
        dec_c = torch.tanh(self.c_proj(c_flat))
        preds = self.decoder(enc_out, dec_h, dec_c, self.H, first_input=first_input)
        return preds

In [ ]:
def minmax_scale(arr, eps=1e-8):
    arr = np.asarray(arr, dtype=np.float32)
    mn = arr.min()
    mx = arr.max()
    denom = mx - mn

    if denom < eps: scaled = np.zeros_like(arr, dtype=np.float32)
    else: scaled = (arr - mn) / (denom)

    return scaled.astype(np.float32), float(mn), float(mx)

class StockDataset(Dataset):
    def __init__(self, name, df):
        self.input_len=INPUT_LENGTH
        self.pred_len=PREDICTIONS_H
        self.name = name

        c_prices = df["Close"].values
        scaled, mn, mx = minmax_scale(c_prices)
        self.scaler = (mn, mx)
        self.prices = torch.tensor(scaled, dtype=torch.float32)
        self.rsi = torch.tensor(df["Rsi"].values,  dtype=torch.float32) / 100
        self.k = torch.tensor(df["%K"].values,  dtype=torch.float32) / 100
        self.d = torch.tensor(df["%D"].values,  dtype=torch.float32) / 100

        self.features = torch.stack([self.prices, self.rsi, self.k, self.d], dim=1)

    def __len__(self):
        return len(self.features) - self.input_len - self.pred_len
    
    def __repr__(self):
        return self.name + f" of len {self.__len__()}"
    
    def __getitem__(self, idx):
        x = self.features[idx:idx+self.input_len]
        y = self.prices[idx+self.input_len:idx+self.input_len+self.pred_len]

        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [ ]:
with open("global_vars.json", 'r') as file:
    vars = json.load(file)

split_ratio = 0.8
files = glob.glob(f"stocks/{vars['PERIOD']}/{vars['INTERVAL']}/*.csv")
train_datasets = []
val_datasets = []

for file in files:
    name = file.split("/")[-1].replace(".csv", "")
    df = pd.read_csv(file)
    split_idx = int(len(df) * split_ratio)

    train_df = df.iloc[:split_idx].reset_index(drop=True)
    validation_df = df.iloc[split_idx:].reset_index(drop=True)

    train_set = StockDataset(name, train_df)
    val_set = StockDataset(name, validation_df)

    # only include datasets that have at least one sample
    if len(train_set) > 0:
        train_datasets.append(train_set)
    if len(val_set) > 0:
        val_datasets.append(val_set)

In [ ]:
from torch.utils.data import ConcatDataset

train_ds = ConcatDataset(train_datasets) if len(train_datasets) > 0 else None
val_ds = ConcatDataset(val_datasets) if len(val_datasets) > 0 else None

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True) if train_ds is not None else None
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False) if val_ds is not None else None

In [ ]:
def prepare_preds_tensor(preds_list):
    preds_tensor = torch.stack(preds_list, dim=1)
    if preds_tensor.size(-1) == 1:
        preds_tensor = preds_tensor.squeeze(-1)
    return preds_tensor

def prepare_target_tensor(y):
    if y.dim() == 3 and y.size(-1) == 1:
        return y.squeeze(-1)
    return y

def stack_preds(preds_list):
    out = torch.stack(preds_list, dim=1) 
    if out.size(-1) == 1: out = out.squeeze(-1)
    return out

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def train_and_validate(model, train_loader, val_loader,
                        epochs=50, weight_decay=0, clip_grad=1.0, save_path="model.pt"):
    model.to(device)
    model.apply(init_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_n = 0

        for x, y in train_loader:
            x = x.to(device).float()
            y = y.to(device).float()

            if x.dim() == 2: x = x.unsqueeze(-1)
            if torch.isnan(x).any() or torch.isinf(x).any() or torch.isnan(y).any() or torch.isinf(y).any():
                continue

            optimizer.zero_grad()
            preds_list = model(x)
            preds = stack_preds(preds_list)

            if preds.dim() == 3 and y.dim() == 2: y_t = y.unsqueeze(-1)
            else: y_t = y

            if torch.isnan(preds).any() or torch.isinf(preds).any():
                continue

            loss = criterion(preds, y_t)
            if not torch.isfinite(loss):
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()

            train_loss += loss.item()
            train_n += 1

        avg_train = train_loss / train_n if train_n > 0 else float("nan")

        # Validation 

        model.eval()
        val_loss = 0.0
        val_n = 0
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device).float()
                y = y.to(device).float()

                if x.dim() == 2: x = x.unsqueeze(-1)
                if torch.isnan(x).any() or torch.isinf(x).any() or torch.isnan(y).any() or torch.isinf(y).any():
                    continue

                preds_list = model(x)
                preds = stack_preds(preds_list)

                if preds.dim() == 3 and y.dim() == 2: y_t = y.unsqueeze(-1)
                else: y_t = y

                if torch.isnan(preds).any() or torch.isinf(preds).any():
                    continue

                loss = criterion(preds, y_t)
                if not torch.isfinite(loss):
                    continue

                val_loss += loss.item()
                val_n += 1

        avg_val = val_loss / val_n if val_n > 0 else float("nan")
        rmse = avg_val**0.5 if avg_val == avg_val and avg_val != float("inf") else float("nan")

        if avg_val == avg_val and avg_val < best_val_loss:
            best_val_loss = avg_val
            print("Saving a new best model... at epoch ", epoch)
            torch.save(model.state_dict(), save_path)

        print(f"Epoch {epoch:3d} | Train MSE: {avg_train:.6f} | Val MSE: {avg_val:.6f} | Val RMSE: {rmse:.6f}")

    print(f"Finished. Best val MSE: {best_val_loss:.6f} (saved to {save_path})")
    return save_path

In [ ]:
model = Wrapper(input_dim=4, enc_emb=32, enc_hid=64, dec_hid=64, attn_dim=32, bidir=True)
save_path = "best_wrapper.pt"

train_and_validate(model,
                   train_loader,
                   val_loader,
                   epochs=50,
                   clip_grad=1.0)

/var/folders/0_/8m1my8650593bn29fpf5hsvc0000gn/T/ipykernel_45558/2463902096.py:39: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


Epoch   1 | Train MSE: 0.004472 | Val MSE: 0.008993 | Val RMSE: 0.094831
Epoch   2 | Train MSE: 0.001640 | Val MSE: 0.007751 | Val RMSE: 0.088037
Epoch   2 | Train MSE: 0.001640 | Val MSE: 0.007751 | Val RMSE: 0.088037
Epoch   3 | Train MSE: 0.001546 | Val MSE: 0.007971 | Val RMSE: 0.089283
Epoch   3 | Train MSE: 0.001546 | Val MSE: 0.007971 | Val RMSE: 0.089283
Epoch   4 | Train MSE: 0.001603 | Val MSE: 0.007356 | Val RMSE: 0.085767
Epoch   4 | Train MSE: 0.001603 | Val MSE: 0.007356 | Val RMSE: 0.085767
Epoch   5 | Train MSE: 0.001542 | Val MSE: 0.007126 | Val RMSE: 0.084415
Epoch   5 | Train MSE: 0.001542 | Val MSE: 0.007126 | Val RMSE: 0.084415
Epoch   6 | Train MSE: 0.001547 | Val MSE: 0.007040 | Val RMSE: 0.083906
Epoch   6 | Train MSE: 0.001547 | Val MSE: 0.007040 | Val RMSE: 0.083906
Epoch   7 | Train MSE: 0.001526 | Val MSE: 0.007326 | Val RMSE: 0.085590
Epoch   7 | Train MSE: 0.001526 | Val MSE: 0.007326 | Val RMSE: 0.085590
Epoch   8 | Train MSE: 0.001496 | Val MSE: 0.007042

'model.pt'